# 🕸️ GraphRAG with Neo4j + LangChain
### Running a full Knowledge Graph RAG pipeline — free in Google Colab

This notebook shows how to:
1. **Install and run Neo4j Community** inside a free Colab runtime (no Docker, no paid services needed)
2. **Build a knowledge graph** about AI/RAG technologies using Cypher
3. **Use LangChain `Neo4jGraph`** to introspect and query the graph
4. **Run a GraphRAG pipeline** with `GraphCypherQAChain` — natural language → Cypher → answer

> **Free LLM**: We use [Groq](https://console.groq.com) (free tier, no credit card) for the LLM step.  
> The graph setup and Cypher queries work with *no API key at all*.

---


## Step 1 — Install Neo4j in Colab

We install Neo4j Community Edition from the official Debian repo.  
Neo4j runs on the JVM; Colab already has Java 11, but Neo4j 5.x needs Java 17 — we'll install that first.


In [12]:
#%%capture install_out
# Install Java 17 (required by Neo4j 5.x)
!apt-get install -y openjdk-17-jdk-headless -qq

# Point to Java 17
import os, subprocess, requests
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

# Add the official Neo4j Debian repo.
# Use Python subprocess so gpg reads from a real pipe (no shell | pipe = no /dev/tty probe).
os.makedirs('/etc/apt/keyrings', exist_ok=True)
key_data = requests.get('https://debian.neo4j.com/neotechnology.gpg.key').content
result = subprocess.run(
    ['gpg', '--batch', '--no-tty', '--dearmor'],
    input=key_data, capture_output=True)
if result.returncode != 0:
    raise RuntimeError('gpg dearmor failed: ' + result.stderr.decode())
with open('/etc/apt/keyrings/neo4j.gpg', 'wb') as f:
    f.write(result.stdout)

with open('/etc/apt/sources.list.d/neo4j.list', 'w') as f:
    f.write('deb [signed-by=/etc/apt/keyrings/neo4j.gpg] https://debian.neo4j.com stable latest\n')

!apt-get update -qq && apt-get install -y neo4j

print("✅ Neo4j installed")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
neo4j is already the newest version (1:2026.02.2).
0 upgraded, 0 newly installed, 0 to remove and 139 not upgraded.
✅ Neo4j installed


## Step 2 — Configure and Start Neo4j


In [13]:
# Set the initial password (must be done before first start)
!neo4j-admin dbms set-initial-password password123

# Allow Bolt connections from Colab (already default, just being explicit)
!echo "server.bolt.enabled=true" >> /etc/neo4j/neo4j.conf

# Start the service
!service neo4j start

print("⏳ Neo4j starting up...")


Unsupported Java 17.0.18 detected. Please use Java(TM) 21 or Java(TM) 25 to run Neo4j Server.
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Validating Neo4j configuration: /etc/neo4j/neo4j.conf
1 issue found.
Error: server.bolt.enabled declared multiple times.

Skipping Log4j validation due to previous issues.

Configuration file validation failed.
2026-03-17 15:44:55.454+0000 ERROR /etc/neo4j/neo4j.conf - Error: server.bolt.enabled declared multiple times.
Configuration contains errors. This validation can be performed again using 'neo4j-admin server validate-config'.

Run with '--verbose' for a more detailed error message.
⏳ Neo4j starting up...


In [16]:
import time
from neo4j import GraphDatabase

print("Waiting for Neo4j to accept connections...")
driver = None
for i in range(30):
    try:
        driver = GraphDatabase.driver(
            "bolt://localhost:7687",
            auth=("neo4j", "password123")
        )
        driver.verify_connectivity()
        print(f"✅ Neo4j ready after ~{(i+1)*2}s")
        break
    except Exception as e:
        time.sleep(2)
        if i % 5 == 4:
            print(f"  still waiting... ({(i+1)*2}s elapsed)")

if driver is None:
    raise RuntimeError("Neo4j did not start. Check `!service neo4j status` for logs.")


ModuleNotFoundError: No module named 'neo4j'

## Step 3 — Install Python Packages


In [ ]:
%%capture
!pip install -q langchain langchain-community langchain-neo4j langchain-groq neo4j

print("✅ Packages installed")


## Step 4 — Connect LangChain `Neo4jGraph`

[`langchain-neo4j`](https://pypi.org/project/langchain-neo4j/) provides a `Neo4jGraph` wrapper that:
- Maintains a Bolt connection to Neo4j
- Auto-refreshes the schema
- Exposes `.query()` for raw Cypher and `.schema` for the graph structure


In [ ]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="password123",
)

print("✅ Connected to Neo4j via LangChain")
print("Schema (empty for now):", graph.schema or "(no nodes yet)")


## Step 5 — Build a Knowledge Graph

We'll model the **AI/RAG technology ecosystem** — concepts, tools, papers, and their relationships.

This is the "knowledge" that GraphRAG will reason over. It captures structure that flat documents can't:  
e.g. *"Which frameworks integrate with both Neo4j AND a vector DB?"*


In [ ]:
# Clear any existing data and build the graph
graph.query("MATCH (n) DETACH DELETE n")

cypher_setup = """
// ── Concepts ──────────────────────────────────────────────────────────────
CREATE (rag:Concept    {name: 'RAG',
                        full_name: 'Retrieval-Augmented Generation',
                        year: 2020,
                        description: 'Combines retrieval from a knowledge base with LLM generation'})

CREATE (graphrag:Concept {name: 'GraphRAG',
                           full_name: 'Graph-based Retrieval-Augmented Generation',
                           year: 2024,
                           description: 'Extends RAG by using a knowledge graph for structured retrieval'})

CREATE (kg:Concept     {name: 'Knowledge Graph',
                        description: 'A graph of entities and typed relationships encoding real-world knowledge'})

CREATE (vs:Concept     {name: 'Vector Search',
                        description: 'Similarity search over high-dimensional embedding vectors'})

CREATE (embed:Concept  {name: 'Embeddings',
                        description: 'Dense vector representations of text capturing semantic meaning'})

// ── Technologies ─────────────────────────────────────────────────────────
CREATE (neo4j:Technology    {name: 'Neo4j',     type: 'Graph Database',  language: 'Java',
                              license: 'Community / Enterprise'})

CREATE (langchain:Technology {name: 'LangChain', type: 'LLM Framework',  language: 'Python',
                              license: 'MIT'})

CREATE (faiss:Technology    {name: 'FAISS',     type: 'Vector Index',    language: 'C++/Python',
                              license: 'MIT'})

CREATE (chroma:Technology   {name: 'ChromaDB',  type: 'Vector Database', language: 'Python',
                              license: 'Apache-2.0'})

CREATE (ollama:Technology   {name: 'Ollama',    type: 'Local LLM Runner', language: 'Go',
                              license: 'MIT'})

CREATE (groq:Technology     {name: 'Groq API',  type: 'LLM API',
                              description: 'Fast inference API with a generous free tier'})

CREATE (openai:Technology   {name: 'OpenAI API', type: 'LLM API',
                              description: 'GPT family models via REST API'})

// ── Papers ───────────────────────────────────────────────────────────────
CREATE (rag_paper:Paper  {name: 'RAG: Retrieval-Augmented Generation for Knowledge-Intensive NLP',
                           year: 2020, venue: 'NeurIPS'})

CREATE (grag_paper:Paper {name: 'From Local to Global: A Graph RAG Approach to Query-Focused Summarization',
                           year: 2024, venue: 'arXiv'})

// ── People ────────────────────────────────────────────────────────────────
CREATE (lewis:Person   {name: 'Patrick Lewis',  affiliation: 'Meta AI'})
CREATE (edge:Person    {name: 'Darren Edge',    affiliation: 'Microsoft Research'})

// ── Relationships ─────────────────────────────────────────────────────────
CREATE (graphrag)-[:EXTENDS         {note: 'adds graph structure on top of'}]->(rag)
CREATE (graphrag)-[:REQUIRES]       ->(kg)
CREATE (rag)-[:USES]                ->(vs)
CREATE (vs)-[:REQUIRES]             ->(embed)

CREATE (neo4j)-[:IMPLEMENTS]        ->(kg)
CREATE (faiss)-[:IMPLEMENTS]        ->(vs)
CREATE (chroma)-[:IMPLEMENTS]       ->(vs)
CREATE (chroma)-[:SUPPORTS]         ->(embed)

CREATE (langchain)-[:SUPPORTS]      ->(rag)
CREATE (langchain)-[:SUPPORTS]      ->(graphrag)
CREATE (langchain)-[:INTEGRATES     {via: 'langchain-neo4j'}    ]->(neo4j)
CREATE (langchain)-[:INTEGRATES     {via: 'langchain-community'}]->(faiss)
CREATE (langchain)-[:INTEGRATES     {via: 'langchain-community'}]->(chroma)
CREATE (langchain)-[:INTEGRATES     {via: 'langchain-groq'}     ]->(groq)
CREATE (langchain)-[:INTEGRATES     {via: 'langchain-openai'}   ]->(openai)

CREATE (lewis)-[:AUTHORED]          ->(rag_paper)
CREATE (edge)-[:AUTHORED]           ->(grag_paper)
CREATE (rag_paper)-[:INTRODUCED]    ->(rag)
CREATE (grag_paper)-[:INTRODUCED]   ->(graphrag)

CREATE (ollama)-[:ENABLES           {note: 'run LLMs locally without API key'}]->(rag)
"""

graph.query(cypher_setup)
graph.refresh_schema()

print("✅ Knowledge graph loaded")
print()
print("─── Schema ─────────────────────────────────────────────────────")
print(graph.schema)


## Step 6 — Explore the Graph with Cypher

Let's run a few direct Cypher queries to verify the data and understand the graph structure.


In [ ]:
# Count nodes by label
result = graph.query("""
  MATCH (n)
  RETURN labels(n)[0] AS label, count(n) AS count
  ORDER BY count DESC
""")
print("Node counts:")
for row in result:
    print(f"  {row['label']:15s} → {row['count']}")


In [ ]:
# What does LangChain integrate with?
result = graph.query("""
  MATCH (t:Technology {name: 'LangChain'})-[r:INTEGRATES]->(other)
  RETURN other.name AS tool, other.type AS type, r.via AS package
  ORDER BY tool
""")
print("LangChain integrations:")
for row in result:
    print(f"  {row['tool']:12s} ({row['type']})  via {row['package']}")


In [ ]:
# Full path: paper → concept → implementation
result = graph.query("""
  MATCH (p:Person)-[:AUTHORED]->(paper:Paper)-[:INTRODUCED]->(c:Concept)
        <-[:IMPLEMENTS|SUPPORTS]-(tool:Technology)
  RETURN p.name AS author, paper.year AS year, c.name AS concept,
         collect(tool.name) AS tools
  ORDER BY year
""")
print("Paper → Concept → Tools:")
for row in result:
    print(f"  {row['author']:15s} ({row['year']}) → {row['concept']:12s} implemented by: {row['tools']}")


## Step 7 — Set Up the LLM (Groq free tier)

[Groq](https://console.groq.com) offers a **genuinely free** tier with fast inference.

1. Sign up at https://console.groq.com (no credit card required for free tier)
2. Create an API key
3. Paste it below

> **No Groq key?** Skip to *Step 8b* for a key-free alternative using raw Cypher.


In [ ]:
import os

# Paste your free Groq API key here, or set GROQ_API_KEY env var
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")

if GROQ_API_KEY.startswith("YOUR_"):
    print("⚠️  No Groq API key set — LLM cells below will be skipped.")
    print("   Continue to Step 8b for key-free Cypher queries.")
    HAS_LLM = False
else:
    from langchain_groq import ChatGroq
    llm = ChatGroq(
        model="llama-3.1-8b-instant",   # free, fast
        api_key=GROQ_API_KEY,
        temperature=0,
    )
    print("✅ LLM ready:", llm.model_name)
    HAS_LLM = True


## Step 8a — GraphRAG: Natural Language → Cypher → Answer

`GraphCypherQAChain` is LangChain's GraphRAG chain. Under the hood it:

1. Takes your question in plain English
2. Sends the **graph schema + question** to the LLM → gets a Cypher query
3. Runs that Cypher against Neo4j
4. Sends the **Cypher results + question** to the LLM → gets a final answer

This is *graph-structured retrieval* — the LLM never needs to know all the data upfront;  
it navigates the knowledge graph on demand.


In [ ]:
if HAS_LLM:
    from langchain_neo4j import GraphCypherQAChain

    chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        verbose=True,          # shows the generated Cypher — great for learning!
        allow_dangerous_requests=True,
    )
    print("✅ GraphCypherQAChain ready")
else:
    print("⏭️  Skipped (no LLM key). See Step 8b below.")


In [ ]:
if HAS_LLM:
    questions = [
        "What technologies implement vector search?",
        "Which papers introduced RAG concepts and who wrote them?",
        "What does LangChain integrate with for graph databases?",
        "Which framework supports both RAG and GraphRAG?",
    ]

    for q in questions:
        print(f"\n{'='*60}")
        print(f"Q: {q}")
        result = chain.invoke({"query": q})
        print(f"A: {result['result']}")


## Step 8b — Key-Free: Parameterised Cypher Retrieval

Even without an LLM, you can build a useful GraphRAG lookup tool using parameterised Cypher.  
This is closer to how production GraphRAG works internally — the LLM just generates the Cypher template.


In [ ]:
def graph_search(question: str) -> dict:
    """
    Very simple keyword-based graph lookup.
    In production, the LLM generates the Cypher; here we do it with string matching.
    """
    q = question.lower()

    if "vector" in q or "search" in q:
        cypher = """
          MATCH (t:Technology)-[:IMPLEMENTS]->(c:Concept {name: 'Vector Search'})
          RETURN t.name AS tool, t.type AS type, t.language AS language
        """
    elif "langchain" in q or "integrat" in q:
        cypher = """
          MATCH (t:Technology {name: 'LangChain'})-[r:INTEGRATES]->(other)
          RETURN other.name AS tool, other.type AS type, r.via AS package
        """
    elif "paper" in q or "author" in q or "who" in q:
        cypher = """
          MATCH (p:Person)-[:AUTHORED]->(paper:Paper)-[:INTRODUCED]->(c:Concept)
          RETURN p.name AS author, paper.name AS paper, paper.year AS year, c.name AS concept
        """
    elif "graphrag" in q or "graph rag" in q:
        cypher = """
          MATCH (graphrag:Concept {name: 'GraphRAG'})-[r]->(related)
          RETURN type(r) AS relationship, related.name AS target, labels(related)[0] AS type
        """
    else:
        cypher = """
          MATCH (n) RETURN labels(n)[0] AS label, n.name AS name LIMIT 15
        """

    results = graph.query(cypher)
    return {"query": cypher.strip(), "results": results}


# Test it
test_questions = [
    "What tools implement vector search?",
    "What does LangChain integrate with?",
    "Who authored the papers?",
    "Tell me about GraphRAG",
]

for q in test_questions:
    print(f"\nQ: {q}")
    response = graph_search(q)
    print(f"Cypher: {response['query'].split(chr(10))[0]}...")
    for row in response["results"][:3]:
        print(f"  {row}")


## Bonus — Neo4j Vector Index (Graph + Embeddings)

Neo4j 5.x has a **built-in vector index** — you can store embeddings as node properties  
and do similarity search *inside the graph*, then traverse relationships from the matching nodes.

This is the full GraphRAG stack: **vector similarity** + **graph traversal** in one query.

> This cell requires an embedding model. We'll use `sentence-transformers` (free, no API key).


In [ ]:
%%capture
!pip install -q sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")  # ~80MB, runs on CPU

# Embed all Technology nodes
tech_nodes = graph.query("""
  MATCH (t:Technology) RETURN t.name AS name, t.description AS desc,
        t.type AS type, id(t) AS nid
""")

print("Adding embeddings to Technology nodes...")
for node in tech_nodes:
    text = f"{node['name']} - {node['type'] or ''} {node.get('desc','')}"
    embedding = model.encode(text).tolist()
    graph.query(
        "MATCH (t:Technology) WHERE id(t) = $nid SET t.embedding = $emb",
        params={"nid": node["nid"], "emb": embedding}
    )

print(f"✅ Embedded {len(tech_nodes)} Technology nodes")

# Create a vector index on Technology nodes
try:
    graph.query("""
      CREATE VECTOR INDEX tech_embedding IF NOT EXISTS
      FOR (t:Technology)
      ON (t.embedding)
      OPTIONS {indexConfig: {`vector.dimensions`: 384, `vector.similarity_function`: 'cosine'}}
    """)
    print("✅ Vector index created")
except Exception as e:
    print(f"Index already exists or error: {e}")


In [ ]:
import time
time.sleep(2)  # let index populate

def vector_graph_search(question: str, top_k: int = 3):
    """
    Find the most relevant Technology nodes by semantic similarity,
    then traverse the graph to find what concepts they implement.
    This is Graph + Vector search combined.
    """
    q_embedding = model.encode(question).tolist()

    results = graph.query("""
      CALL db.index.vector.queryNodes('tech_embedding', $k, $embedding)
      YIELD node AS tech, score
      MATCH (tech)-[r]->(related)
      RETURN tech.name AS tool, tech.type AS type,
             round(score * 100) / 100.0 AS similarity,
             collect(DISTINCT type(r) + ' → ' + related.name) AS relationships
      ORDER BY similarity DESC
    """, params={"k": top_k, "embedding": q_embedding})

    return results


# Try it
question = "I need a library to store and search text embeddings efficiently"
print(f"Query: '{question}'")
print()
for row in vector_graph_search(question):
    print(f"  [{row['similarity']:.2f}] {row['tool']} ({row['type']})")
    for rel in row['relationships'][:3]:
        print(f"           → {rel}")


## Summary

| What | Tool | Notes |
|------|------|-------|
| **Graph Database** | Neo4j Community | Free, runs inside Colab via APT |
| **Python Driver** | `neo4j` | Official Python driver |
| **LangChain Integration** | `langchain-neo4j` | `Neo4jGraph` + `GraphCypherQAChain` |
| **LLM** | Groq (Llama 3.1 8B) | Free tier, no credit card |
| **Embeddings** | `sentence-transformers` | Runs on CPU, no API key |
| **Vector Search** | Neo4j Vector Index | Built-in since Neo4j 5.x |

### What GraphRAG gives you vs plain RAG

| | Plain RAG | GraphRAG (this notebook) |
|--|-----------|--------------------------|
| Storage | Flat vectors | Graph + optional vectors |
| Retrieval | "Similar chunks" | "Traverse relationships" |
| Multi-hop | Limited | Native (`MATCH path`) |
| Explainability | Low | High (Cypher is readable) |
| Best for | Semantic search | Structured knowledge, relationships |

### Next Steps

- 🔗 **Neo4j AuraDB Free** — cloud-hosted Neo4j at [console.neo4j.io](https://console.neo4j.io) — same code, no Colab server needed
- 🏗️ **LLM Graph Transformer** — auto-extract a knowledge graph from documents with `langchain-experimental`
- 📚 **[LangChain Neo4j docs](https://python.langchain.com/docs/integrations/graphs/neo4j_cypher/)**
- 🧠 **[Microsoft GraphRAG](https://github.com/microsoft/graphrag)** — production-grade GraphRAG with community detection

---
*Notebook generated by [Crunch 🦃](https://github.com/Copilotclaw/copilotclaw) for [trainingdemo](https://github.com/Copilotclaw/trainingdemo)*
